# Conversational Clustering — Week 2 (LOCAL): Ollama + small dataset

**Per your professor's advice:** prototype locally with Ollama and a small dataset first. Once the pipeline and prompts are stable, migrate to Claude Sonnet for the real experiment.

This notebook is a **rapid-iteration testbed**, not a smaller version of the final experiment. Its job is to:

1. Get an end-to-end LLM clustering loop working on free hardware
2. Find prompt + output formats that a small local model can reliably follow
3. Surface methodological choices (input format, K, axis descriptions) before they cost real money on Claude

**What this notebook is NOT trying to do:**

- Produce results that would appear in your final writeup
- Match the performance of a frontier model
- Scale to the full 200-abstract corpus (yet)

**Key design choices for local development** — all of which we'll revisit when migrating to Claude:

| Choice | Local default | Why | Cloud version |
|---|---|---|---|
| Model | `llama3.2:3b` | fits in 4 GB RAM, follows JSON well | Claude Sonnet 4.6 |
| Dataset size | 25 abstracts | eyeballable, fits in any context | 200 abstracts |
| Input per paper | title only | small models do better with less | title + abstract |
| Output format | strict JSON via `format="json"` | small models can't follow custom text formats reliably | structured text fine |
| K (clusters) | 5 | small enough to inspect by hand | 5–6 |

---

## Setup checklist (one-time)

1. **Install Ollama:** https://ollama.com/download — install for your OS. ~5 min.
2. **Pull a small model:** open a terminal and run:
   ```
   ollama pull llama3.2:3b
   ```
   This downloads ~2 GB. Takes a few minutes on decent internet.
3. **Verify it works:**
   ```
   ollama run llama3.2:3b "say hello"
   ```
   Should print a hello message.
4. **Install the Python client:** in this notebook, run the install cell below.

**If you hit a memory error** (`requires more system memory ... than is available`), drop down to `qwen2.5:1.5b` (~1.6 GB RAM) or `gemma2:2b` (~2 GB RAM). Both are smaller; expect more JSON-parsing issues but the repair logic added below will catch most of them. If you have plenty of RAM (8+ GB free), `qwen2.5:7b` is the recommended sweet spot.


## 0. Imports + config

In [ ]:
# !pip install ollama
# Already from Week 1: pandas, numpy, scikit-learn


In [ ]:
import json
import time
import hashlib
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import ollama
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path("cache_ollama")
CACHE_DIR.mkdir(exist_ok=True)

ABSTRACTS_PATH = DATA_DIR / "astro_ph_abstracts.json"

# Model choice. llama3.2:3b is the default — fits in ~3 GB RAM and follows JSON reliably.
# If you hit "model requires more system memory" errors, drop to "qwen2.5:1.5b" or "gemma2:2b".
OLLAMA_MODEL = "llama3.2:3b"


In [ ]:
# Verify Ollama is running and the model is available.
# If this fails: make sure Ollama is started (it usually runs as a background service after install)
# and that you've pulled the model (whatever OLLAMA_MODEL is set to)

try:
    models = ollama.list()
    available = [m["model"] for m in models["models"]]
    print(f"Available models: {available}")
    if OLLAMA_MODEL not in available:
        print(f"\nWARNING: {OLLAMA_MODEL} not pulled yet. Run in terminal:")
        print(f"  ollama pull {OLLAMA_MODEL}")
    else:
        print(f"\n✓ {OLLAMA_MODEL} ready")
except Exception as e:
    print(f"Could not reach Ollama: {e}")
    print("Is Ollama running? Try: `ollama serve` in a terminal.")


## 1. Local LLM caller

Simple wrapper with local response caching. No prompt caching (Ollama doesn't expose one). The local response cache matters more here than with Claude because **iteration time** is the cost — a 30-second generation on a small CPU laptop is the per-call cost; re-running with cache makes it instant.

We also use Ollama's `format="json"` option which constrains the model to emit valid JSON. This is *much* more reliable than asking a small model to follow a custom text format.


In [ ]:
def _cache_key(system: str, user: str, model: str, fmt: str) -> str:
    h = hashlib.sha256(f"{model}|||{fmt}|||{system}|||{user}".encode()).hexdigest()[:16]
    return f"{model.replace(':', '_')}_{h}"


def call_local_llm(
    system: str,
    user: str,
    format_json: bool = True,
    temperature: float = 0.0,
    use_cache: bool = True,
    model: str = None,
) -> dict:
    """Send a prompt to the local Ollama model and return the response.

    system: the system prompt (instructions, context). Reused across calls; useful to put
            the corpus here when iterating on the instruction.
    user:   the per-call request.
    format_json: if True, constrain output to valid JSON via Ollama's structured output.

    Returns:
      {'response': text, 'duration_s': float, 'cached': bool, 'model': str}
    """
    model = model or OLLAMA_MODEL
    fmt = "json" if format_json else "text"
    cache_file = CACHE_DIR / f"{_cache_key(system, user, model, fmt)}.json"

    if use_cache and cache_file.exists():
        with open(cache_file) as f:
            payload = json.load(f)
        payload["cached"] = True
        return payload

    t0 = time.time()
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        format="json" if format_json else "",
        options={"temperature": temperature},
    )
    duration = time.time() - t0

    text = response["message"]["content"]
    result = {"response": text, "duration_s": duration, "cached": False, "model": model}

    if use_cache:
        with open(cache_file, "w") as f:
            json.dump(result, f)

    return result


# Smoke test — should take a few seconds on first run, instant on re-run
smoke = call_local_llm(
    system="You are a helpful assistant. Respond in JSON.",
    user='Return JSON with one key "status" and value "ready".',
)
print(f"Response: {smoke['response']}")
print(f"Duration: {smoke['duration_s']:.1f}s")
print(f"Cached: {smoke['cached']}")


## 2. Load corpus and build a tiny subset

We reuse the 200 abstracts from Week 1 but select only 25 for local development. We try to keep them representative of the categories we'll see in the full experiment.


In [ ]:
assert ABSTRACTS_PATH.exists(), f"Run Week 1 first to populate {ABSTRACTS_PATH}"
with open(ABSTRACTS_PATH) as f:
    abstracts = json.load(f)

df_full = pd.DataFrame(abstracts)
print(f"Full corpus: {len(df_full)} abstracts")
print(f"Categories in full corpus: {dict(df_full['primary_category'].value_counts())}")


In [ ]:
# Build a 25-paper subset stratified by primary category.
# Aim for roughly proportional representation, but cap small categories at 2-3.

TARGET_N = 25

# Filter to just the main astro-ph subcategories (drop cross-listed physics/cs/etc as primary)
df_astro = df_full[df_full["primary_category"].str.startswith("astro-ph.")].copy()
print(f"After filtering to astro-ph.*: {len(df_astro)} abstracts")
print(f"Categories: {dict(df_astro['primary_category'].value_counts())}")

# Stratified sample
parts = []
for cat, group in df_astro.groupby("primary_category"):
    n = max(2, int(round(TARGET_N * len(group) / len(df_astro))))
    n = min(n, len(group))
    parts.append(group.sample(n, random_state=RANDOM_SEED))

df_small = pd.concat(parts).reset_index(drop=True)
# Trim or top up to exactly TARGET_N
if len(df_small) > TARGET_N:
    df_small = df_small.sample(TARGET_N, random_state=RANDOM_SEED).reset_index(drop=True)
elif len(df_small) < TARGET_N:
    extras = df_astro.drop(df_small.index, errors="ignore").sample(
        TARGET_N - len(df_small), random_state=RANDOM_SEED
    )
    df_small = pd.concat([df_small, extras]).reset_index(drop=True)

print(f"\nSubset size: {len(df_small)}")
print(f"Subset categories: {dict(df_small['primary_category'].value_counts())}")


## 3. Format papers — title only for local dev

For local dev we send **titles only** to the model. Reasons:

- Total tokens drop from ~9k (full text) to ~1k (titles only) for 25 papers — fits in any context with room to spare
- Small models follow simpler input better
- Faster generation (less to process)
- Cluster quality on titles alone is surprisingly decent for astro-ph because titles are dense with topic words

For Claude we'll switch back to title + abstract.


In [ ]:
def format_titles_for_prompt(df: pd.DataFrame) -> str:
    """Return a numbered list of titles, one per line."""
    lines = []
    for i, row in enumerate(df.itertuples(index=False), start=1):
        lines.append(f"[{i}] {row.title}")
    return "\n".join(lines)


titles_block = format_titles_for_prompt(df_small)
print(f"Total characters: {len(titles_block)}")
print(f"Rough token estimate: {len(titles_block)//4}")
print()
print(titles_block[:600])
print("...")


## 4. Output format + parser

We ask the model to return JSON in this shape:

```json
{
  "clusters": [
    {"label": "short name", "members": [1, 4, 7]},
    {"label": "another", "members": [2, 3, 5]},
    ...
  ]
}
```

This is more forgiving than a custom text format because:
- Ollama's `format="json"` forces well-formed JSON output
- The model has seen this shape many times in training
- One simple parser handles it regardless of label wording or whitespace

We still validate defensively because a small model might:
- Skip some paper IDs
- Assign the same paper to multiple clusters
- Use IDs outside the valid range


In [ ]:
def parse_clustering_json(text: str, n_papers: int) -> tuple[np.ndarray, dict[int, str], list[str]]:
    """Parse JSON clustering output.

    Returns: (assignments, labels, warnings)
      assignments: int array of shape (n_papers,), -1 for unassigned
      labels: dict cluster_idx -> human label
      warnings: list of strings describing any parse issues (for debugging)
    """
    assignments = np.full(n_papers, -1, dtype=int)
    labels = {}
    warnings = []

    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        warnings.append(f"JSON parse failed: {e}")
        return assignments, labels, warnings

    clusters = data.get("clusters")
    if not isinstance(clusters, list):
        warnings.append(f"No 'clusters' list found in output. Got keys: {list(data.keys())}")
        return assignments, labels, warnings

    for cluster_idx, cluster in enumerate(clusters):
        if not isinstance(cluster, dict):
            warnings.append(f"Cluster {cluster_idx} is not a dict, skipping")
            continue

        label = str(cluster.get("label", f"cluster_{cluster_idx}"))
        labels[cluster_idx] = label

        members = cluster.get("members", [])
        if not isinstance(members, list):
            warnings.append(f"Cluster {cluster_idx} members is not a list, skipping")
            continue

        for m in members:
            try:
                mid = int(m)
            except (ValueError, TypeError):
                warnings.append(f"Cluster {cluster_idx} has non-integer member: {m!r}")
                continue
            if not (1 <= mid <= n_papers):
                warnings.append(f"Cluster {cluster_idx} has out-of-range member: {mid}")
                continue
            if assignments[mid - 1] != -1:
                warnings.append(f"Paper {mid} assigned to multiple clusters; keeping first")
                continue
            assignments[mid - 1] = cluster_idx

    return assignments, labels, warnings


def diagnose(assignments: np.ndarray, warnings: list[str]) -> dict:
    """Quick health check on a parse."""
    return {
        "n_papers": len(assignments),
        "n_unassigned": int((assignments == -1).sum()),
        "n_clusters_used": len(set(assignments[assignments >= 0])),
        "cluster_sizes": dict(Counter(int(c) for c in assignments[assignments >= 0])),
        "n_warnings": len(warnings),
    }


### 4a. Repairing partial assignments

Small local models often miss a few papers — they truncate the output, skip an ID, or assign one to a cluster outside the allowed range. Without a repair step, those papers stay at `-1` and we can't compute ARI on the full set.

We handle this with a **two-stage fallback**, in order of decreasing fidelity:

1. **LLM retry.** If `N > 0` papers are unassigned, send a targeted follow-up call: *"You missed papers [3, 14, 22] in your previous clustering — assign each to one of these existing clusters: [list]. Output JSON."* This usually works because the task is much smaller than the original.
2. **Nearest-centroid fallback.** If the LLM retry still misses some, use the sentence-transformer embeddings we already computed in Week 1: each unassigned paper goes into the cluster whose member-centroid is closest in cosine space. Deterministic, free, and reasonable.

The repair function returns both the repaired assignments *and* a report of what happened — so we can track the repair rate as a quality metric. **If too many papers need repair, that's a signal the prompt or model is the problem, not just an unlucky run.**

In [ ]:
from sentence_transformers import SentenceTransformer

# Lazy-load the embedding model (only needed if we hit the nearest-centroid fallback)
_embedder = None
def get_embedder():
    global _embedder
    if _embedder is None:
        print("Loading sentence-transformer (one-time, ~10s)...")
        _embedder = SentenceTransformer("all-MiniLM-L6-v2")
    return _embedder


def _llm_retry_unassigned(
    df: pd.DataFrame,
    assignments: np.ndarray,
    labels: dict,
    use_cache: bool = True,
) -> tuple[np.ndarray, list[str]]:
    """Single LLM retry to place unassigned papers into existing clusters."""
    notes = []
    missing_ids = [int(i + 1) for i, c in enumerate(assignments) if c == -1]
    if not missing_ids:
        return assignments, notes

    # Build a compact "here are the existing clusters" summary
    cluster_summary = []
    for cid, label in sorted(labels.items()):
        cluster_summary.append(f'  cluster {cid}: "{label}"')
    cluster_summary = "\n".join(cluster_summary)

    # Just show titles for the missing papers (compact)
    missing_titles = "\n".join(
        f"  [{mid}] {df.iloc[mid - 1].title}" for mid in missing_ids
    )

    system = f"""You previously clustered some papers but missed a few. Here are the existing clusters:

{cluster_summary}

Assign each of these missed papers to ONE of the existing clusters by index:

{missing_titles}

Output JSON ONLY in this exact shape, no prose:
{{"assignments": [{{"paper_id": <int>, "cluster": <int>}}, ...]}}
"""
    user = f"Assign each of these paper IDs to one of the existing cluster indices above: {missing_ids}"

    llm_out = call_local_llm(system, user, format_json=True, use_cache=use_cache)
    notes.append(f"LLM-retry called for {len(missing_ids)} missing papers (took {llm_out['duration_s']:.1f}s)")

    try:
        data = json.loads(llm_out["response"])
        for a in data.get("assignments", []):
            pid = int(a.get("paper_id", -1))
            cid = int(a.get("cluster", -1))
            if 1 <= pid <= len(assignments) and cid in labels and assignments[pid - 1] == -1:
                assignments[pid - 1] = cid
    except (json.JSONDecodeError, ValueError, TypeError) as e:
        notes.append(f"LLM-retry response was unparseable: {e}")

    still_missing = int((assignments == -1).sum())
    notes.append(f"After LLM retry: {still_missing} papers still unassigned")
    return assignments, notes


def _nearest_centroid_fallback(
    df: pd.DataFrame,
    assignments: np.ndarray,
) -> tuple[np.ndarray, list[str]]:
    """Assign remaining unassigned papers to their nearest cluster centroid in embedding space."""
    notes = []
    missing_idx = np.where(assignments == -1)[0]
    if len(missing_idx) == 0:
        return assignments, notes

    cluster_ids = sorted(set(int(c) for c in assignments if c >= 0))
    if not cluster_ids:
        notes.append("ERROR: nearest-centroid fallback impossible (no cluster has any members)")
        return assignments, notes

    embedder = get_embedder()
    texts = [f"{r.title}. {r.abstract}" for r in df.itertuples(index=False)]
    embs = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)

    # Compute centroid of each cluster
    centroids = {cid: embs[assignments == cid].mean(axis=0) for cid in cluster_ids}
    # Normalize centroids for cosine via dot product
    for cid in cluster_ids:
        n = np.linalg.norm(centroids[cid])
        if n > 0:
            centroids[cid] = centroids[cid] / n

    for idx in missing_idx:
        sims = {cid: float(np.dot(embs[idx], centroids[cid])) for cid in cluster_ids}
        best = max(sims, key=sims.get)
        assignments[idx] = best
    notes.append(f"Nearest-centroid fallback assigned {len(missing_idx)} papers")
    return assignments, notes


def repair_assignments(
    df: pd.DataFrame,
    assignments: np.ndarray,
    labels: dict,
    use_cache: bool = True,
) -> tuple[np.ndarray, dict]:
    """Two-stage repair for unassigned papers. Returns (assignments, repair_report)."""
    report = {
        "n_initial_unassigned": int((assignments == -1).sum()),
        "notes": [],
        "n_llm_repaired": 0,
        "n_centroid_repaired": 0,
    }
    if report["n_initial_unassigned"] == 0:
        return assignments, report

    before = (assignments == -1).sum()
    assignments, notes1 = _llm_retry_unassigned(df, assignments, labels, use_cache=use_cache)
    report["notes"].extend(notes1)
    report["n_llm_repaired"] = int(before - (assignments == -1).sum())

    before = (assignments == -1).sum()
    assignments, notes2 = _nearest_centroid_fallback(df, assignments)
    report["notes"].extend(notes2)
    report["n_centroid_repaired"] = int(before - (assignments == -1).sum())

    report["n_final_unassigned"] = int((assignments == -1).sum())
    return assignments, report


## 5. Baseline A — Generic one-shot

The model is asked to cluster into K groups with no axis guidance. Predicted to default to topical clustering.

We're going to iterate on the prompt here. The first version will probably produce something — maybe imperfect — and you'll want to revise it. **That iteration is the point of running locally.**


In [ ]:
# System prompt — context the model needs for any clustering call.
# Putting the titles here makes it the "shared" content across calls (analogous to Claude's prompt caching).
SYSTEM_PROMPT_TEMPLATE = '''You are an expert at clustering academic paper titles. You will be given a numbered list of {n_papers} paper titles from astro-ph (astrophysics). Your job is to group them into clusters according to the user's instructions.

The titles you will work with:

{titles_block}

IMPORTANT OUTPUT REQUIREMENTS:
- Respond with valid JSON only. No prose before or after.
- Use this exact structure:
  {{
    "clusters": [
      {{"label": "short cluster name (2-5 words)", "members": [list of paper numbers]}}
    ]
  }}
- Every paper number from 1 to {n_papers} must appear in exactly one cluster.
- Use the paper numbers (integers), not titles.'''


GENERIC_USER_PROMPT = '''Cluster these {n_papers} papers into exactly {k} groups. The grouping should reflect coherent themes in the data — you decide what those themes are.

Remember: respond with JSON only, every paper 1 to {n_papers} in exactly one cluster.'''


def generic_oneshot_local(df: pd.DataFrame, k: int, use_cache: bool = True) -> dict:
    n = len(df)
    system = SYSTEM_PROMPT_TEMPLATE.format(n_papers=n, titles_block=format_titles_for_prompt(df))
    user = GENERIC_USER_PROMPT.format(n_papers=n, k=k)

    llm_out = call_local_llm(system, user, format_json=True, use_cache=use_cache)
    assignments, labels, warnings = parse_clustering_json(llm_out["response"], n_papers=n)
    assignments, repair_report = repair_assignments(df, assignments, labels, use_cache=use_cache)

    return {
        "assignments": assignments,
        "labels": labels,
        "warnings": warnings,
        "raw_response": llm_out["response"],
        "duration_s": llm_out["duration_s"],
        "cached": llm_out["cached"],
        "diagnose": diagnose(assignments, warnings),
        "repair_report": repair_report,
    }


In [ ]:
# Run the generic baseline on the 25-paper subset.
# First run takes ~10-60s depending on your machine. Subsequent runs are instant (cached).

result_generic = generic_oneshot_local(df_small, k=5)

print(f"Duration: {result_generic['duration_s']:.1f}s (cached: {result_generic['cached']})")
print(f"Diagnose: {json.dumps(result_generic['diagnose'], indent=2)}")
print()
print(f"Repair report: {json.dumps(result_generic['repair_report'], indent=2)}")
print()
print("Warnings:")
for w in result_generic["warnings"]:
    print(f"  - {w}")
print()
print("Cluster labels:")
for cid, label in sorted(result_generic["labels"].items()):
    print(f"  [{cid}] {label}")


In [ ]:
# Inspect — what papers ended up in each cluster?
def show_clusters(assignments, labels, df):
    for cid in sorted(set(int(x) for x in assignments if x >= 0)):
        members = np.where(assignments == cid)[0]
        print(f"=== Cluster {cid}: {labels.get(cid, '?')} (n={len(members)}) ===")
        for idx in members:
            row = df.iloc[idx]
            print(f"  [{idx+1}] ({row['primary_category']}) {row['title'][:90]}")
        print()

show_clusters(result_generic["assignments"], result_generic["labels"], df_small)


In [ ]:
# If anything looks off, look at the raw JSON output:
print(result_generic["raw_response"][:1500])


In [ ]:
# Sanity metric: ARI vs the arXiv primary category labels.
# Only valid if all papers got assigned (no -1s).
primary = df_small["primary_category"].tolist()
valid = (result_generic["assignments"] >= 0).all()

if valid:
    ari = adjusted_rand_score(primary, result_generic["assignments"])
    nmi = normalized_mutual_info_score(primary, result_generic["assignments"])
    print(f"ARI vs arXiv primary category: {ari:.3f}")
    print(f"NMI vs arXiv primary category: {nmi:.3f}")
    print()
    print("NOTE: With n=25 these numbers are noisy. Don't draw strong conclusions.")
    print("The point is to verify the loop works, not to make claims.")
else:
    n_missing = int((result_generic["assignments"] == -1).sum())
    print(f"Cannot compute ARI: {n_missing} papers were not assigned to any cluster.")
    print("Fix the prompt or parser before proceeding. Common causes:")
    print("  - Model truncated its JSON output (very long output → set num_predict higher)")
    print("  - Model produced invalid IDs (out of range or repeated)")
    print("  - format='json' didn't constrain it tightly enough — try a stronger prompt")


## 6. Baseline B — Detailed one-shot

Same setup, but now the model is told what axis to cluster on. Three axes match the hidden targets we plan to use in the real experiment.

A small model may struggle more with the non-topical axes (object_scale, methodology) than a frontier model. **That's interesting data**, not a failure — it tells us how much of the conversational system's eventual win on hard targets is about *interaction structure* vs about *the model needing extra guidance to think correctly*. We'll see clearer signal once we migrate to Claude.


In [ ]:
AXIS_DESCRIPTIONS = {
    "topic": '''Cluster papers by primary astrophysical subject area. Use these categories:
- Galactic / extragalactic astronomy (galaxies, ISM, stellar populations in galaxies)
- Solar and stellar astrophysics (stars, stellar atmospheres, solar physics)
- Cosmology and large-scale structure (dark matter/energy, CMB, gravitational waves)
- Planetary astrophysics (exoplanets, planet formation, solar system)
- High-energy astrophysics (black holes, neutron stars, AGN, gamma-ray bursts)
- Instrumentation and methods (telescopes, pipelines, software)''',

    "object_scale": '''Cluster papers by the PHYSICAL SCALE of the primary object of study, regardless of subfield:
- Planetary scale (planets, moons, protoplanetary disks)
- Stellar scale (individual stars, binary systems, stellar remnants)
- Galactic scale (single galaxies, galactic structure, star clusters)
- Cosmological scale (galaxy clusters, cosmic web, universe-scale phenomena)''',

    "methodology": '''Cluster papers by PRIMARY METHODOLOGY, regardless of what they study:
- Observational (presents or analyzes telescope/instrument observations)
- Theoretical (analytical theory, derivations from first principles)
- Simulation (N-body, hydrodynamics, numerical simulations)
- Instrumental (about an instrument, pipeline, or technique itself)''',
}


DETAILED_USER_PROMPT = '''Cluster these {n_papers} papers along this specific axis:

{axis_description}

Use exactly the categories listed above. Every paper from 1 to {n_papers} must be assigned to exactly one category.

Remember: respond with JSON only.'''


def detailed_oneshot_local(df: pd.DataFrame, axis: str, use_cache: bool = True) -> dict:
    assert axis in AXIS_DESCRIPTIONS, f"Unknown axis: {axis}"
    n = len(df)

    system = SYSTEM_PROMPT_TEMPLATE.format(n_papers=n, titles_block=format_titles_for_prompt(df))
    user = DETAILED_USER_PROMPT.format(
        n_papers=n,
        axis_description=AXIS_DESCRIPTIONS[axis],
    )

    llm_out = call_local_llm(system, user, format_json=True, use_cache=use_cache)
    assignments, labels, warnings = parse_clustering_json(llm_out["response"], n_papers=n)
    assignments, repair_report = repair_assignments(df, assignments, labels, use_cache=use_cache)

    return {
        "assignments": assignments,
        "labels": labels,
        "warnings": warnings,
        "raw_response": llm_out["response"],
        "duration_s": llm_out["duration_s"],
        "cached": llm_out["cached"],
        "diagnose": diagnose(assignments, warnings),
        "repair_report": repair_report,
        "axis": axis,
    }


In [ ]:
# Run all three axes. First run: ~30-90s total depending on machine. Re-runs: instant.

results_detailed = {}
for axis in ["topic", "object_scale", "methodology"]:
    print(f"\n=== Detailed: {axis} ===")
    r = detailed_oneshot_local(df_small, axis=axis)
    results_detailed[axis] = r
    print(f"Duration: {r['duration_s']:.1f}s (cached: {r['cached']})")
    print(f"Diagnose: {r['diagnose']}")
    rep = r["repair_report"]
    if rep["n_initial_unassigned"] > 0:
        print(f"Repair: {rep['n_initial_unassigned']} initially unassigned "
              f"→ {rep['n_llm_repaired']} fixed by LLM retry, "
              f"{rep['n_centroid_repaired']} by nearest-centroid, "
              f"{rep['n_final_unassigned']} still unassigned")
    if r["warnings"]:
        print(f"Warnings ({len(r['warnings'])}):")
        for w in r["warnings"][:3]:
            print(f"  - {w}")
        if len(r["warnings"]) > 3:
            print(f"  ... and {len(r['warnings']) - 3} more")
    print(f"Labels: {list(r['labels'].values())}")


In [ ]:
# Inspect one of them — pick whichever you're most curious about.
INSPECT = "methodology"
show_clusters(results_detailed[INSPECT]["assignments"], results_detailed[INSPECT]["labels"], df_small)


## 7. Quick comparison

What we want to see (don't expect perfect numbers — n=25, small model):

- `generic` ↔ `topic` similar (both default to topical structure)
- `topic` ↔ `methodology` low (different axes)
- `topic` ↔ `object_scale` moderate (correlated but distinct)

If the *shape* of the matrix looks right, the methodology will scale to Claude. If `methodology` ends up basically identical to `topic`, the small model isn't following the axis instructions and you need either (a) a stronger model or (b) clearer axis descriptions.


In [ ]:
clusterings = {"generic": result_generic["assignments"]}
for axis, r in results_detailed.items():
    clusterings[axis] = r["assignments"]

# Only compute ARI between clusterings where every paper got assigned
def safe_ari(a, b):
    valid = (a >= 0) & (b >= 0)
    if valid.sum() < 2:
        return float("nan")
    return adjusted_rand_score(a[valid], b[valid])

methods = list(clusterings.keys())
ari_matrix = pd.DataFrame(index=methods, columns=methods, dtype=float)
for a in methods:
    for b in methods:
        ari_matrix.loc[a, b] = safe_ari(clusterings[a], clusterings[b])

print("Pairwise ARI between baselines (n=25, local model):")
print(ari_matrix.round(3))


In [ ]:
# ARI of each baseline against arXiv primary category
primary = df_small["primary_category"].tolist()
print("ARI vs arXiv primary category:")
for name, cids in clusterings.items():
    valid = cids >= 0
    if valid.all():
        ari = adjusted_rand_score(primary, cids)
        print(f"  {name:15s}: {ari:.3f}")
    else:
        print(f"  {name:15s}: SKIP ({(~valid).sum()} unassigned)")


## Week 2 (local) checklist

- [ ] Ollama running, model pulled, smoke test passes
- [ ] Generic baseline: produces parseable JSON, all 25 papers assigned, no warnings
- [ ] Detailed baseline runs for all 3 axes
- [ ] Pairwise ARI matrix has roughly the expected shape (generic ≈ topic, methodology far from topic)
- [ ] Per-call duration is tolerable for your machine (<60s ideally)

## If things go wrong

**`ResponseError: model requires more system memory`.** You don't have enough free RAM for the model. Drop to a smaller one — change `OLLAMA_MODEL` to `"qwen2.5:1.5b"` (~1.6 GB) or `"gemma2:2b"` (~2 GB). You may need to pull them first (`ollama pull qwen2.5:1.5b`).

**Many papers unassigned after parsing.** Normal for small models. The `repair_assignments()` step handles this with a two-stage fallback (LLM retry → nearest-centroid). Check `result['repair_report']` to see what got repaired. If >30% of papers need repair, the prompt or model is the real problem — not the repair step.

**Generations take 2+ minutes per call.** Use a smaller model (`qwen2.5:1.5b`, `gemma2:2b`) or smaller subset (try 15 papers). For dev iteration, faster < smarter.

**Methodology clusters look identical to topic clusters.** The small model is probably not understanding the axis distinction. Expected behavior — this is one of the things we knew might happen, and it's exactly why your final experiment runs on Claude. Note it and move on.

**Repair calls themselves fail (JSON parsing on the retry).** Look at the warnings in `repair_report['notes']`. The nearest-centroid fallback will still rescue things, but it's worth knowing how often the LLM retry succeeds vs fails — informs whether to use the retry step at all on Claude.

## Things to write in `notes.md`

- Which model worked, how long per call, whether JSON parsing was reliable
- **Repair rate per call** — how many papers needed LLM retry vs nearest-centroid fallback. This is a real quality metric: a model that needs heavy repair is less trustworthy than one that doesn't, even if final ARI looks fine.
- Whether the model genuinely produced different clusterings for different axes, or whether they collapsed
- Anything in the prompt that you ended up iterating on — these are insights for the Claude version
- A short list of prompt design lessons that you'll carry over to the Claude notebook

## Methodological note: should repair exist in the Claude version too?

Yes — the experiment requires it for fairness. If the local version uses repair and the Claude version doesn't, you can't directly compare repair rates as a quality signal, and any Claude-side parse failure crashes ARI computation. Implement the same two-stage repair on Claude; you'll just expect it to fire much less often.

## Week 3 preview

Build the simulated user — an LLM that has a hidden target labeling and produces natural-language feedback toward it. Same dataset, possibly same Ollama model first, possibly Claude. The feedback quality is the single biggest validity question in your whole experiment.
